<h2> Waves Unit, Q3  (CIEM3000)</h2>

<h1 style="color:#00BFFF;"> Some useful functions </h1>

A number of functions were already provided for the computer labs in Q2 to calculate wave statistics based on both time-domain and frequency-domain approaches. Below you can find a few additional ones that may come in handy when analysing your data. 

**Decomposition of incoming and reflected waves from colocated measurements of velocity and elevation**

The method by Guza, as described in the background information provided in Topic 4. allows to separate incoming and reflected surface elevation timeseries from the total (measured) elevation and velocity timeseries provided that these are collocated (same x-position). However, it also assumes that waves linear and in shallow water, i.e. that all wave components of the wave field propagate at the same speed, which is c= sqrt(g*depth).

The function provided below (`colocated_decomposition()`) is also based on linear wave theory, but does not assume that waves are in shallow water. It uses (general) linear wave theory to link the elevation to the velocity signals. The underlying concepts are described in Appendix B of the paper by Buckley et al. 2015 (Dynamics of Wave Setup over a Steeply Sloping Fringing Reef, https://journals-ametsoc-org.tudelft.idm.oclc.org/view/journals/phoc/45/12/jpo-d-15-0067.1.xml).

Main inputs are (check function for more details):

- Co-located surface elevation and velocity timeseries (same cross-shore position)
- The height above the bed at which the velocity measurement is performed
- The sampling frequency of the signal
- And some sort of threshold value that is used to avoid problems at high frequencies where velocity signals are typically weak, and noise may dominate (we advise you to use a value of 0.05 for this one)

The function `colocated_decomposition()` calls the function `disper()` (also defined below) to calculate the wave number based on linear wave theory. Feel free to replace `disper` by your own function created earlier in the course. Note that `colocated_decomposition` requires as input the velocity at a given elevation (and not a depth-averaged value). 

In [1]:
import numpy as np

def disper(w, h):
    # Approximation of the linear dispersion relation with an absolute error in k*h < 5.0e-16 for all k*h; 
    #   Input:
    # w = 2*pi/T, were T is wave period (Hz)
    # h = water depth (m)
    #  Output:
    #   k = wave number (m^-1)
    

    w2 = (w**2) * h / g
    q = w2 / (1 - np.exp(-w2**(5/4)))**(2/5)

    for j in range(2):
        thq = np.tanh(q)
        thq2 = 1 - thq**2
        a = (1 - q * thq) * thq2
        b = thq + q * thq2
        c = q * thq - w2
        arg = (b**2) - 4 * a * c
        arg = (-b + np.sqrt(arg)) / (2 * a)
        iq = np.where(np.abs(a*c) < 1.0e-8 * (b**2))
        arg[iq] = -c[iq] / b[iq]
        q = q + arg

    k = np.sign(w) * q / h
    ik = np.isnan(k)
    k[ik] = np.zeros_like(k[ik])

    return k



def colocated_decomposition(t, eta, u, h, zu, cutoff=None):
    # Frequency domain algorithm (assuming linear wave theory) to separate shoreward-propagating and seaward-propagating wave motion
    # from co-located velocity and free surface elevation measurements.
    # Inputs:
    #   t - time vector (s)
    #   eta - timeseries of free surface elevation  (m)
    #   u   - timeseries of velocity in x-direction (=direction of wave propagation) (m/s)
    #   zu  - height above the bed at which the velocity u has been measured (m)
    #   cutoff -  minimum value for the linear wave theory velocity response function Ku that links elevation to horizontal velocity (-). 
    #             This threshold is used to define a lower bound for Ku to avoid "blowing up noise" in the high-frequency part of the signal. Unless specified otherwise, cutoff=0.05. 
    # Outputs:
    #    eta_i - free surface elevation timeseries for the incident (shoreward-propagating) waves (m)
    #    eta_r - free surface elevation timeseries for the reflected (seaward-propagating) waves (m)
    #    u_i - velocity timeseries for the incident (shoreward-propagating) waves (m/s)
    #    u_r - velocity timeseries for the reflected (seaward-propagating) waves (m/s)
    # Author (original Matlab version): Jochem Dekkers; Based on the methodology described in the paper by Buckley et al. (2015), Dynamics of Wave Setup over a Steeply Sloping Fringing Reef (see appendix)
    
    if cutoff is None:
        cutoff = 0.05  # default value below which the velocity response function Ku is 'cutoff';

    g = 9.81
    dt = t[1] - t[0]
    nfft = len(t)
    D = nfft * dt
    fs = 1 / dt
    df = 1 / D
    f = np.arange(0, fs, df)
    fneg = np.arange(nfft // 2 + 1, nfft)
    fpos = np.arange(1, nfft // 2 + 1)
    
    Feta = np.fft.fft(eta, nfft) / nfft
    Fu = np.fft.fft(u, nfft) / nfft
    
    k = disper(2 * np.pi * f[fpos], h)
    Ku = np.cosh(k * zu) / np.cosh(k * h)
    cosinehyp = k * h
    ii = np.argmax(Ku < cutoff)
    Ku[ii+1:] = cutoff
    
    Fetai = 1 / 2 * (Feta[fpos] + 2 * np.pi * f[fpos] / (g * k * Ku) * Fu[fpos])
    Fetai[ii+1:] = 0
    Fetaitot = np.zeros(nfft, dtype=complex)
    Fetaitot[fpos] = Fetai
    Fetaitot[fneg] = np.flip(np.conj(Fetaitot[fpos]))
    eta_i = np.real(np.fft.ifft(Fetaitot, nfft) * nfft)
    
    Fui = g * k / (2 * np.pi * f[fpos]) * Ku * Fetai
    Fui[ii+1:] = 0
    Fuitot = np.zeros(nfft, dtype=complex)
    Fuitot[fpos] = Fui
    Fuitot[fneg] = np.flip(np.conj(Fuitot[fpos]))
    u_i = np.real(np.fft.ifft(Fuitot, nfft) * nfft)
    
    Fetar = 1 / 2 * (Feta[fpos] - 2 * np.pi * f[fpos] / (g * k * Ku) * Fu[fpos])
    Fetar[ii+1:] = 0
    Fetartot = np.zeros(nfft, dtype=complex)
    Fetartot[fpos] = Fetar
    Fetartot[fneg] = np.flip(np.conj(Fetartot[fpos]))
    eta_r = np.real(np.fft.ifft(Fetartot, nfft) * nfft)
    
    Fur = -g * k / (2 * np.pi * f[fpos]) * Ku * Fetar
    Fur[ii+1:] = 0
    Furtot = np.zeros(nfft, dtype=complex)
    Furtot[fpos] = Fur
    Furtot[fneg] = np.flip(np.conj(Furtot[fpos]))
    u_r = np.real(np.fft.ifft(Furtot, nfft) * nfft)

    return eta_i, eta_r, u_i, u_r


**Basic band-pass frequency filter**

In [2]:
import numpy as np

def frequency_filter(data, Fs, f_low, f_high):
    ''' frequency_filter is a simple spectral filter in which the unwanted frequencies (below f_low and above f_high) 
        are set to zero before coming back to the time-domain
            input: data timeseries you want to filter
                   F_s the sampling frequency of this timeseries (Hz)
                   f_low and f_high are the limits of the band pass filter (Hz)
            output: data_filtered band pass filtered timeseries (same unit as the input timeseries)
    '''
    
    N = len(data)

    fft_data = np.fft.fft(data)  # fourier transform of the signal

    freq_vector = np.fft.fftfreq(N, d=1/Fs) # corresponding (2-sided) frequency axis (includes positive and negative values)
    
    idx = np.where((abs(freq_vector) > f_high) | (abs(freq_vector) <= f_low)) # we select the indices to filter out
    
    fft_data[idx]=0.  # we set the the fourier coefficients corresponding to abs(f)>f_high and abs(f)<f_low to zero

    data_filtered = np.fft.ifft(fft_data).real  # we come back to the time domain with an inverse Fourier transform
    
    return data_filtered